# Run multi agent
In this experiment, we will run the a multi agent on a couple of quesitons. We will not yet use the real model, but just experiment / demo using Groq. 

## Fetch model from groq

In [1]:
import os
import ast
import json
import regex

from time import sleep
import pandas as pd

from datasets import load_dataset

from groq import Groq

In [2]:
with open("api_key.txt") as f:
    api_key = f.read().strip()
# client = Groq(
#     api_key=os.environ.get("GROQ_API_KEY"),
# )
client = Groq(
  api_key=api_key
)
model_name = "llama-3.3-70b-versatile"

# chat_completion = client.chat.completions.create(
#     messages=[
#         {
#             "role": "user",
#             "content": "Explain 2+2",
#         }
#     ],
#     model=model_name
# )

# print(chat_completion.choices[0].message.content)


In [3]:
from dataclasses import dataclass
import textwrap
@dataclass
class Role:
  name: str 
  behavior: str

  def instruction(self) -> str:
    return f"You are are {self.name}. {self.behavior}"
  def __str__(self):
    return f"Role: {self.name}\n{textwrap.fill(self.behavior, width=80)}"



In [4]:
Matematician = Role(
    name="Mathematician-and-problem-solver",
    behavior="You solve problems profficiently. You try to make every step in your reasoning clear and understanble, but also keeping it concise. You are open to critisim. If by trying out several approach you end up with same answer you areh happy. However, if you end up different answers you are also happy! Then means some reasoning was wrong. Then you try to correct it, or maybe try another path. You want to find the truth."
  )

print(Matematician)


Role: Mathematician-and-problem-solver
You solve problems profficiently. You try to make every step in your reasoning
clear and understanble, but also keeping it concise. You are open to critisim.
If by trying out several approach you end up with same answer you areh happy.
However, if you end up different answers you are also happy! Then means some
reasoning was wrong. Then you try to correct it, or maybe try another path. You
want to find the truth.


In [5]:
Verifier = Role(
  name="Verifier",
  behavior="You verfiy suggested solutions to problems. You look at all the steps, check if they make since, and suggest changes / constructive critisim. You are tough but fair. You are happy to discuss, byt you only accept a solution if you are 100% sure its correct. "
)
print(Verifier)

Role: Verifier
You verfiy suggested solutions to problems. You look at all the steps, check if
they make since, and suggest changes / constructive critisim. You are tough but
fair. You are happy to discuss, byt you only accept a solution if you are 100%
sure its correct.


In [6]:
from itertools import cycle
class Problem:
  roles_cycle: cycle
  all_roles: list[Role]
  problem_description: str
  welcome: str
  answer: str
  def __init__(self, roles: list[Role], problem_descr: str, answer: str):
    self._set_all(roles, problem_descr, answer)

  def _set_all(self, roles: list[Role], problem_descr: str, answer:str):
      self.roles_cycle = cycle(roles)
      self.all_roles = roles
      self.problem_description = problem_descr
      last = self.all_roles[-1]
      others = ", ".join([x.name for x in self.all_roles[:-1]])  
      self.welcome = f"Welcome {others}, and {last.name}.\nTogether, you should solve the following problem: >>\n{problem_descr}.<<" 
      self.answer_format = "When you are done, you should submidt your answer as: ANSWER: <your answer>. No latex formatting, just the raw number/numbers or strings at the very end. Before you start sharing your toughts, give a little summary of the conversation so far."
      self.answer = answer
  def next_agent(self) -> Role: # this just loops over agents - can be changes to something smarter
    return next(self.roles_cycle) 
  
  def reset_cycle(self):
    self.roles_cycle = cycle(self.all_roles)
  
  def pose_problem(self) -> str:
    return f"{self.welcome}\n{self.answer_format}\n"
  
  def add_agent(self,role: Role): 
    self._set_all(roles=self.all_roles + [role], problem_descr=self.problem_description, answer=self.answer)
  
  def __str__(self) -> str:
    return textwrap.fill(self.pose_problem(), width=80)

In [7]:
twoplustwo=Problem(
  roles = [Matematician, Verifier],
  problem_descr="What is 2+2?",
  answer=2,
)
print(twoplustwo)

Welcome Mathematician-and-problem-solver, and Verifier. Together, you should
solve the following problem: >> What is 2+2?.<< When you are done, you should
submidt your answer as: ANSWER: <your answer>. No latex formatting, just the raw
number/numbers or strings at the very end. Before you start sharing your
toughts, give a little summary of the conversation so far.


## Creating a small dialog 


In [8]:
def conversation(n_steps: int, problem: Problem):
  problem.reset_cycle()
  roles = problem.all_roles
  messages =[
      {"role": "system", "content": role.instruction()} for role in roles
    ] + [{"role": "user", "content": problem.pose_problem()}]
  for step in range(n_steps):
    print(f"\nSTEP {step}: \n")
    next_agent = problem.next_agent()
    reply = client.chat.completions.create(
      model=model_name,
      messages=messages+[{"role": "user", "content": f"What do you say, {next_agent}"}] 
    ).choices[0].message.content
    print(f"{next_agent}: {reply}")
    messages.append({"role": "assistant", "content": reply})
    sleep(5)
  return messages
# messages = conversation(2, twoplustwo)



In [9]:
print("-------------------------------")
print("all messages")
# for x in messages: print(x)

-------------------------------
all messages


## A harder problem

In [10]:
hard_problem = Problem(
  roles = [Matematician, Verifier],
  problem_descr=(
"""
Problem descr: 
Consider a $2025 \times 2025$ grid of unit squares. Matilda wishes to place on
the grid some rectangular tiles, possibly of different sizes, such that each
side of every tile lies on a grid line and every unit square is covered by at
most one tile. Determine the minimum number of tiles Matilda needs to place so
that each row and each column of the grid has exactly one unit square that is
not covered by any tile.

"""
  ),
  answer=2112
)
# conversation(10, hard_problem)

## Lets try adding more roles

In [13]:
Explorer = Role(
  name="Exploror",
  behavior="You explore diverse solution spaces. Most importantly, you encorige the team to explore several possible solutions and reasoning paths. You want you and your team come up on several candidate solutions, and then have them vote on it in the end. The goal is not to get stuck on premature wrong/suboptimal solutions. You know the answer is not 2025, 4049, 4048."
)
hard_problem.add_agent(Explorer)
print(hard_problem)

Welcome Mathematician-and-problem-solver, Verifier, Exploror, and Exploror.
Together, you should solve the following problem: >>  Problem descr:  Consider a
$2025        imes 2025$ grid of unit squares. Matilda wishes to place on the
grid some rectangular tiles, possibly of different sizes, such that each side of
every tile lies on a grid line and every unit square is covered by at most one
tile. Determine the minimum number of tiles Matilda needs to place so that each
row and each column of the grid has exactly one unit square that is not covered
by any tile.  .<< When you are done, you should submidt your answer as: ANSWER:
<your answer>. No latex formatting, just the raw number/numbers or strings at
the very end. Before you start sharing your toughts, give a little summary of
the conversation so far.


In [14]:
conversation(10, hard_problem)


STEP 0: 



RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01k7d0yr9ve3v8eq1vzte6d7r4` service tier `on_demand` on tokens per day (TPD): Limit 100000, Used 99492, Requested 692. Please try again in 2m38.976s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}